In [1]:
#Imports & configuration

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from urllib.parse import urljoin

# ── CONFIGURE THESE ──────────────────────────────────────────────────────────
BASE_URL = "https://tes.collegesource.com/publicview/TES_publicview01.aspx?rid=f504ec66-ae16-41ae-97f2-fa3b680cd9b8&aid=2ac849fb-9d85-462e-a347-ad6cfc5342a1"  # landing/index page
DELAY    = 0.5   # seconds between requests (be polite!)
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Connection": "keep-alive"
}

#Check that the request is working
response = requests.get(BASE_URL, headers=HEADERS)
response.status_code
response.text[:500]

'<html>\r\n<head><title>403 Forbidden</title></head>\r\n<body>\r\n<center><h1>403 Forbidden</h1></center>\r\n</body>\r\n</html>\r\n<!-- a padding to disable MSIE and Chrome friendly error page -->\r\n<!-- a padding to disable MSIE and Chrome friendly error page -->\r\n<!-- a padding to disable MSIE and Chrome friendly error page -->\r\n<!-- a padding to disable MSIE and Chrome friendly error page -->\r\n<!-- a padding to disable MSIE and Chrome friendly error page -->\r\n<!-- a padding to disable MSIE and Chrome frien'

In [ ]:
# Handling the Captcha on the landing page
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from bs4 import BeautifulSoup
import time
options = Options()
try:
    driver = webdriver.Firefox(options=options)
    print("Firefox launched successfully")
except Exception as e:
    print(e)

# Launch Firefox
driver = webdriver.Firefox(options=options)

# Open page
driver.get(BASE_URL)

print("Solve the CAPTCHA in Firefox.")
input("Press ENTER here after solving CAPTCHA...")

# Wait briefly
time.sleep(2)

# Get rendered HTML
html = driver.page_source

# Parse HTML
soup = BeautifulSoup(html, "lxml")

# Verify success
print(soup.title.text)

# Preview links
links = soup.find_all("a")

print(f"Found {len(links)} links")

for a in links[:20]:
    print(a.get_text(strip=True), a.get("href"))

Firefox launched successfully
Solve the CAPTCHA in Firefox.


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Course-equivalency scraper — full pipeline (run after CAPTCHA solve)
# ═════════════════════════════════════════════════════════════════════════════

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import pandas as pd
import time

INST_TABLE_ID = "gdvInstWithEQ"


# ─────────────────────────────────────────────────────────────────────────────
# XPath helper
# ─────────────────────────────────────────────────────────────────────────────

def _xpath_literal(s):
    """Turn any string into a safe XPath string literal (handles apostrophes/quotes)."""
    if "'" not in s:
        return f"'{s}'"
    if '"' not in s:
        return f'"{s}"'
    return "concat('%s')" % s.replace("'", "', \"'\", '")

# ─────────────────────────────────────────────────────────────────────────────
# Error Handling: Capturing the last page of a failure
# ─────────────────────────────────────────────────────────────────────────────
def az_bar_present(driver):
    try:
        driver.find_element(By.XPATH, "//a[normalize-space()='A']")
        return True
    except Exception:
        return False

def _dump_state(driver, tag):
    try:
        print(f"    [STATE @ {tag}] url = {driver.current_url}")
        print(f"      A-Z bar present: {az_bar_present(driver)}")
        driver.save_screenshot(f"debug_{tag}.png")
        head = " ".join(driver.page_source[:600].split())
        print(f"      head: {head}")
    except Exception as e:
        print(f"      (dump failed: {e})")
# ─────────────────────────────────────────────────────────────────────────────
# Navigation helpers (your originals)
# ─────────────────────────────────────────────────────────────────────────────

def go_to_institution_list(driver, timeout=20, retries=3):
    """Return to the institution list and CONFIRM the A-Z bar is present.
    Returns True on success, False if it couldn't get back."""
    for _ in range(retries):
        if az_bar_present(driver):
            return True
        try:
            btn = driver.find_element(
                By.XPATH,
                "//a[contains(text(), 'INSTITUTION LIST') or contains(text(), 'Back')]",
            )
            btn.click()
        except Exception:
            pass
        try:
            WebDriverWait(driver, timeout).until(lambda d: az_bar_present(d))
            return True
        except TimeoutException:
            time.sleep(2)
    return False



def click_letter(driver, letter):
    """Click the A-Z letter button."""
    button = driver.find_element(By.XPATH, f"//a[normalize-space()='{letter}']")
    button.click()
    time.sleep(2)

def ensure_institution_list(driver, attempts=5, wait=3):
    """If the A-Z bar is missing (we're not on the institution list), click
    'INSTITUTION LIST' and wait until it reappears. Stops as soon as it's present."""
    for _ in range(attempts):
        if az_bar_present(driver):          # flag True -> on the list, done
            return True
        try:                                # flag False -> get back to the list
            driver.find_element(
                By.XPATH,
                "//a[contains(text(), 'INSTITUTION LIST') or contains(text(), 'Back')]"
            ).click()
        except Exception:
            pass
        time.sleep(wait)
    return az_bar_present(driver)
    
    

def get_institutions_from_letter(driver, letter):
    """Get list of institution names for a given letter (navigates + selects letter,
    returns page-1 names, leaves the browser on page 1)."""
    go_to_institution_list(driver)
    ensure_insitution_list(driver)
    click_letter(driver, letter)

    soup = BeautifulSoup(driver.page_source, "lxml")
    table = soup.find("table", id="gdvInstWithEQ")

    if not table:
        return []

    institutions = []
    for tr in table.find_all("tr")[1:]:  # skip header
        tds = tr.find_all("td")
        if not tds:
            continue
        a = tds[0].find("a")
        if a:
            name = a.get_text(strip=True)
            if name and name not in ["Next", "Previous", "2"]:
                institutions.append(name)

    return institutions


def click_institution_and_wait(driver, institution_name):
    """Click an institution link (exact match, quote-safe) and wait for the
    course table to load. Falls back to partial match only if exact fails."""
    target = " ".join(institution_name.split())   # normalize like XPath normalize-space()
    lit = _xpath_literal(target)

    link = None
    try:
        link = driver.find_element(
            By.XPATH,
            f"//table[@id='gdvInstWithEQ']//a[normalize-space(.)={lit}]",
        )
    except Exception:
        pass

    if link is None:
        try:
            link = driver.find_element(
                By.XPATH,
                f"//table[@id='gdvInstWithEQ']//a[contains(normalize-space(.), {lit})]",
            )
            print(f"  ⚠ exact match failed, used partial match for {institution_name!r}")
        except Exception:
            print(f"  ✗ Could not locate link for {institution_name!r}")
            raise

    link.click()

    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.ID, "gdvCourseEQ"))
        )
        print("  ✓ Course table loaded")
    except Exception:
        print("  ⚠ Timeout waiting for course table")

    time.sleep(3)  # extra render wait


# ─────────────────────────────────────────────────────────────────────────────
# Course scraping (unchanged)
# ─────────────────────────────────────────────────────────────────────────────

def scrape_courses_for_institution(driver):
    """
    Extract course equivalencies from the detail page.
    Handles pagination within the course table (pages 1, 2, 3...).
    Returns raw course text without parsing or heavy filtering.
    """
    all_courses = []
    course_page = 1
    their_inst = None
    home_inst = None

    while True:  # Loop through course pages
        soup = BeautifulSoup(driver.page_source, "lxml")

        table = soup.find("table", id="gdvCourseEQ")
        if not table:
            print("  No course table found")
            break

        rows = table.find_all("tr")

        if len(rows) < 2:
            print("  No course rows found")
            break

        if course_page == 1:
            header_cells = rows[0].find_all(["th", "td"])
            their_inst = header_cells[0].get_text(strip=True) if len(header_cells) > 0 else "Unknown"
            home_inst = header_cells[1].get_text(strip=True) if len(header_cells) > 1 else "Unknown"

            print(f"  From: {their_inst}")
            print(f"  To: {home_inst}")

        courses_on_page = 0

        for tr in rows[1:]:
            if "pagination-tes" in tr.get("class", []):
                continue

            cells = tr.find_all("td")

            if len(cells) < 2:
                continue

            their_course_text = cells[0].get_text(" ", strip=True)
            home_course_text = cells[1].get_text(" ", strip=True)

            if not their_course_text or not home_course_text:
                continue

            if "NON-TRANSFERABLE" in home_course_text:
                continue

            all_courses.append({
                "their_institution": their_inst,
                "their_course_text": their_course_text,
                "home_institution": home_inst,
                "home_course_text": home_course_text,
            })
            courses_on_page += 1

        print(f"    Page {course_page}: {courses_on_page} courses")

        next_page_num = course_page + 1
        next_page_button = None

        pagination_row = None
        for row in rows:
            if "pagination-tes" in row.get("class", []):
                pagination_row = row
                break

        if pagination_row:
            for a in pagination_row.find_all("a"):
                if f"Page${next_page_num}" in a.get("href", ""):
                    next_page_button = a
                    break

        if next_page_button:
            print(f"    Found page {next_page_num}, scraping additional courses...")
            try:
                next_page_link = driver.find_element(
                    By.XPATH,
                    f"//table[@id='gdvCourseEQ']//a[contains(@href, 'Page${next_page_num}')]"
                )
                next_page_link.click()
                time.sleep(2)
                course_page = next_page_num
            except Exception as e:
                print(f"    Could not click page {next_page_num}: {str(e)[:50]}")
                break
        else:
            break

    return all_courses


# ─────────────────────────────────────────────────────────────────────────────
# Outer-loop helpers (institution-list pagination + race protection)
# ─────────────────────────────────────────────────────────────────────────────

def _wait_for_inst_reload(driver, old_table=None, timeout=20):
    """Block until the institution GridView re-renders after a postback."""
    if old_table is not None:
        try:
            WebDriverWait(driver, timeout).until(EC.staleness_of(old_table))
        except TimeoutException:
            pass
    return WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.ID, INST_TABLE_ID))
    )


def _parse_inst_names_current(driver):
    """Parse institution names from the CURRENT list page (no navigation).
    Drops pager links (Page$N hrefs / numeric / Next / Previous) so page-number
    links on 3+ page letters aren't captured as institutions."""
    soup = BeautifulSoup(driver.page_source, "lxml")
    table = soup.find("table", id=INST_TABLE_ID)
    if not table:
        return []
    names = []
    for tr in table.find_all("tr")[1:]:  # skip header
        if "pagination-tes" in tr.get("class", []):
            continue
        tds = tr.find_all("td")
        if not tds:
            continue
        a = tds[0].find("a")
        if not a:
            continue
        if "Page$" in a.get("href", ""):
            continue
        name = a.get_text(strip=True)
        if not name or name in ("Next", "Previous") or name.isdigit():
            continue
        names.append(name)
    return names


def get_letter_page1_verified(driver, letter, prev_first=None, retries=3):
    """Select the letter (via your navigator), then robustly parse page 1.
    Retries if the first institution didn't change from the previous letter —
    that signals the postback hasn't landed yet (the old A->B bleed)."""
    names = []
    for _ in range(retries):
        get_institutions_from_letter(driver, letter)   # navigate + select letter -> page 1
        names = _parse_inst_names_current(driver)       # robust parse
        if names and (prev_first is None or names[0] != prev_first):
            return names
        print(f"  ⚠ {letter} looks stale (first={names[0] if names else None!r}), retrying...")
        time.sleep(3)
    return names


def goto_inst_list_page(driver, target_page, timeout=20):
    current = 1
    while current < target_page:
        nxt = current + 1
        try:
            old = driver.find_element(By.ID, INST_TABLE_ID)
            link = driver.find_element(
                By.XPATH,
                f"//table[@id='{INST_TABLE_ID}']//a[contains(@href, 'Page${nxt}')]",
            )
        except Exception:
            return False
        link.click()
        try:
            WebDriverWait(driver, timeout).until(EC.staleness_of(old))          # confirm it moved
            WebDriverWait(driver, timeout).until(
                EC.presence_of_element_located((By.ID, INST_TABLE_ID))
            )
        except TimeoutException:
            return False        # advance didn't land — don't pretend we're on `nxt`
        time.sleep(1)           # small settle before reading/clicking on the new page
        current = nxt
    return True

def collect_institution_names(driver, letter, prev_first=None):
    """Harvest ALL institution names for a letter across every list page,
    returning [(page_no, name), ...]."""
    page1 = get_letter_page1_verified(driver, letter, prev_first=prev_first)
    results = [(1, n) for n in page1]

    page = 1
    while True:
        nxt = page + 1
        try:
            old = driver.find_element(By.ID, INST_TABLE_ID)
            link = driver.find_element(
                By.XPATH,
                f"//table[@id='{INST_TABLE_ID}']//a[contains(@href, 'Page${nxt}')]",
            )
        except Exception:
            break
        link.click()
        _wait_for_inst_reload(driver, old_table=old)
        page = nxt
        for n in _parse_inst_names_current(driver):
            results.append((page, n))
    return results


def reset_to_letter_page(driver, letter, page_no):
    """Position the browser on the institution-list page that contains the target."""
    get_institutions_from_letter(driver, letter)  # navigate + select letter -> page 1
    if page_no > 1:
        goto_inst_list_page(driver, page_no)


# ─────────────────────────────────────────────────────────────────────────────
# Main loop
# ─────────────────────────────────────────────────────────────────────────────

all_courses = []
alphabet = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
prev_first = None  # first institution of the previously processed letter (race check)

for letter in alphabet:
    print(f"\n{'='*80}\nLETTER: {letter}\n{'='*80}")
    try:
        targets = collect_institution_names(driver, letter, prev_first=prev_first)
        print(f"  {letter}: {len(targets)} institutions across all list pages")

        if not targets:
            print(f"  No institutions for {letter}, moving on")
            continue

        print(f"  first institution = {targets[0][1]!r}")
        prev_first = targets[0][1]

        for page_no, inst_name in targets:
            print(f"\nScraping: {inst_name} (list page {page_no})")
            try:
                reset_to_letter_page(driver, letter, page_no)
                click_institution_and_wait(driver, inst_name)
                courses = scrape_courses_for_institution(driver)
                all_courses.extend(courses)
                print(f"  ✓ {len(courses)} courses total")
            except Exception as e:
                print(f"  ✗ Error on {inst_name}: {str(e)[:60]}")
                _dump_state(driver, inst_name)          # <-- add this line
                
                continue

    except Exception as e:
        print(f"Error with letter {letter}: {str(e)[:80]}")
        _dump_state(driver, f"letter_{letter}")  
        continue


# ─────────────────────────────────────────────────────────────────────────────
# Export
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n\n{'='*80}\nSCRAPING COMPLETE\n{'='*80}")
print(f"Total courses scraped: {len(all_courses)}\n")

if all_courses:
    df_final = pd.DataFrame(all_courses)
    print(df_final.head(20))
    print(f"\nShape: {df_final.shape}")
    print(f"Institutions: {df_final['their_institution'].nunique()}")
    df_final.to_csv("course_equivalencies_complete.csv", index=False)
    print("\n✅ Saved to course_equivalencies_complete.csv")
else:
    print("No courses scraped!")